# IRT experiment — PISA 2018 science

This notebook estimates a **latent factor** and **direct item-to-item links (a graph S)** in item response data with local dependence, using the FLaG-IRT model (Chen et al., 2018).
It applies the model to real data, the PISA 2018 science assessment (data: OECD, 2019; technical details: OECD, 2020; 2,711 students × 102 items, 71% missing). As an exploratory experiment, it also fits a VAE-based IRT model (VAE-MIRT; Curi et al., 2019; Wu et al., 2020; Urban & Bauer, 2021) and a variant of it with the same graph term (called VIF here), and compares them with FLaG-IRT on held-out prediction and on the estimated graph.

## Models

All models handle binary responses $x_{ij}\in\{0,1\}$ (student $i$, item $j$).
$S$ is a symmetric item × item matrix with zero diagonal; $s_{jk}\neq 0$ means there is an edge (local dependence) between items $j$ and $k$.

| Model | Equation | Handling of theta | Role |
|---|---|---|---|
| **FLaG-IRT** (Chen et al., 2018) | $M=A\Sigma A^\top+S$, $\;\mathrm{logit}\,P(x_{ij}=1\mid x_{i,-j})=\sum_{k\neq j}M_{jk}x_{ik}+M_{jj}/2$ | integrated out (pseudo-likelihood) | Established model |
| **FLaG-nograph** | FLaG-IRT with off-diagonal $S=0$ | same | warm start for FLaG |
| **VIF-model** | $\mathrm{logit}\,P(x_{ij}=1)=a_j\theta_i-d_j+\sum_{k\neq j}s_{jk}x_{ik}$ | estimated by an encoder (amortized variational inference) | Experimental model |
| **MIRT-only** (VAE-MIRT) | VIF-model with $S=0$ | same | graph-free baseline · warm start for VIF |

* **FLaG-IRT** — integrating theta out gives an Ising-type model with item interaction matrix $M=A\Sigma A^\top+S$ (Chen et al., 2018). The sum of conditional log-likelihoods (pseudo-likelihood; Besag, 1975) plus an L1 penalty on $S$ (Tibshirani, 1996) is minimized with Adam (Kingma & Ba, 2015) for the smooth parameters and a soft-thresholding (proximal; Parikh & Boyd, 2014) step for $S$.
  For comparison, the fit is mapped to the same representation $(A, d, S, \theta)$ as VIF ($\theta$ is a linear score of the responses).
* **VIF-model** — the same graph term added to a VAE-type IRT model (Kingma & Welling, 2014). An encoder (two-layer tanh MLP) takes the responses and the observation indicators `[X, mask]` and outputs $q(\theta_i\mid x_i)=N(\mu_i,\sigma_i^2)$; the prior is $N(0,1)$.
  The objective is $Q=(E_q[\mathrm{NLL}]+\mathrm{KL})/N+\gamma\lVert s\rVert_1$, optimized with the same Adam + soft-thresholding scheme as FLaG.

## Arms compared

PISA has no true graph, so the arms differ in how the graph is chosen.

| Arm | Graph |
|---|---|
| `FLaG (known graph)`, `VIF-model (known graph)` | all pairs within each of the 6 item clusters are allowed; unpenalized fit |
| `FLaG (unknown graph)` | L1-regularized path over a $\gamma$ grid → unpenalized refit for each support → support with **minimum BIC** |
| `VIF-model (unknown graph)` | VIF-model refitted with the support selected by FLaG's BIC as the given graph |
| `VIF-model (own BIC)` | refit on the minimum-BIC support from VIF-model's own $\gamma$ path (for comparison) |
| `MIRT-only` | no graph (VAE-MIRT baseline) |

## Evaluation

* Train/test is an 85 / 15 split **at the cell level**. Every model is trained on the train cells only and evaluated by **AUC / NLL per cell** on the test cells.
* The split is redrawn `PISA_N_SEEDS` times, all arms are refitted, and mean ± SD is reported (data and fitting seeds are fixed).
* Overlap between the estimated S and the item clusters (`S_precision`: share of selected edges that fall inside a cluster).
  **Caution**: clusters coincide with booklet blocks, so items in the same cluster are observed together. Selected edges falling inside clusters is therefore
  not by itself evidence of local dependence. Section 5 includes a co-observation diagnostic, but whether the selected edges reflect
  actual local dependence beyond the booklet design is still being investigated; here it is reported **as a diagnostic only**, and no conclusion is drawn from it.
* CPW identifiability index `cpw_ident = inc × mu` (Chandrasekaran, Parrilo & Willsky, 2012): the larger it is, the more easily the low-rank part $A$ and the sparse part $S$ can substitute for each other.

## Implementation details

* $A$ is initialized at $|N(0,1)|$ in every model (starts positive; no positivity constraint). Graph models start from the graph-free solution, with $S$ starting at 0.
* The graph-free VIF-model solution (MIRT-only) is the one with the lowest training objective $Q$ among 5 fits that differ only in seed (to avoid local solutions where theta collapses).
* Early stopping is based on convergence of the training objective on the train cells (for VIF, $Q$ is computed with 5 MC draws fixed at the start of the fit).
* BIC $=2\,\mathrm{NLL}+(\text{number of parameters})\log N$ (Schwarz, 1978); the $\gamma$-path + BIC selection follows Chen et al. (2018). The fit term is the marginal NLL with theta integrated out by quadrature for VIF-model, and the objective itself for FLaG.
  The support of a regularized fit is the set of pairs with $|s_{jk}|>0.01$.

## References

* Besag, J. (1975). Statistical analysis of non-lattice data. *Journal of the Royal Statistical Society: Series D (The Statistician)*, 24(3). https://doi.org/10.2307/2987782
* Chandrasekaran, V., Parrilo, P. A., & Willsky, A. S. (2012). Latent variable graphical model selection via convex optimization. *The Annals of Statistics*, 40(4). https://doi.org/10.1214/11-AOS949
* Chen, Y., Li, X., Liu, J., & Ying, Z. (2018). Robust measurement via a fused latent and graphical item response theory model. *Psychometrika*, 83(3). https://doi.org/10.1007/s11336-018-9610-4
* Curi, M., Converse, G. A., Hajewski, J., & Oliveira, S. (2019). Interpretable variational autoencoders for cognitive models. *2019 International Joint Conference on Neural Networks (IJCNN)*. https://doi.org/10.1109/IJCNN.2019.8852333
* Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1412.6980
* Kingma, D. P., & Welling, M. (2014). Auto-encoding variational Bayes. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1312.6114
* OECD (2019). *PISA 2018 Database*. https://www.oecd.org/pisa/data/2018database/
* OECD (2020). *PISA 2018 Technical Report*. OECD Publishing, Paris.
* Parikh, N., & Boyd, S. (2014). Proximal algorithms. *Foundations and Trends in Optimization*, 1(3). https://doi.org/10.1561/2400000003
* Schwarz, G. (1978). Estimating the dimension of a model. *The Annals of Statistics*, 6(2). https://doi.org/10.1214/aos/1176344136
* Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B*, 58(1). https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
* Urban, C. J., & Bauer, D. J. (2021). A deep learning algorithm for high-dimensional exploratory item factor analysis. *Psychometrika*, 86(1). https://doi.org/10.1007/s11336-021-09748-3
* Wu, M., Davis, R. L., Domingue, B. W., Piech, C., & Goodman, N. (2020). Variational item response theory: Fast, accurate, and expressive. *Proceedings of the 13th International Conference on Educational Data Mining (EDM)*. https://arxiv.org/abs/2002.00276

## How to run

**Environment**: Python 3.10+, `torch`, `numpy`, `pandas`, `matplotlib`, `joblib` (CPU only).

```bash
pip install torch numpy pandas matplotlib joblib
```

**Data**: the CSV at `DATA_PATH` has **rows = items (102), columns = students (2,711)**, with student IDs in the header row. Values are 0 / 1, missing entries are empty (NaN).
Items must be ordered by cluster (items 1–17, 18–35, 36–52, 53–68, 69–83, 84–102).

**Runtime**: about 35 min for VIF-model and 25 min for FLaG per split (on 32 cores; varies with the number of cores). The number of workers can be set with the `IRT_WORKERS` environment variable.
We recommend first setting `SMOKE = True` to check that a small configuration runs end to end, then running the full experiment with `SMOKE = False`.

Set the path and the number of repetitions in the cell below.

In [ ]:
DATA_PATH = "data/modified data(science 2018).csv"   # CSV with rows = items, columns = students
SMOKE = False          # True runs a small configuration end to end (for checking the code)

PISA_N_SEEDS = 2       # number of times the train/test split is redrawn
PISA_N_ITER = 5000     # max iterations per fit (early stopping usually kicks in before this)
if SMOKE:
    PISA_N_ITER, PISA_N_SEEDS = 300, 2

# **1. Libraries**

In [ ]:
import os
import warnings

# CPU only, so hide the CUDA initialization warning torch emits when probing for a GPU (workers: _worker_init)
warnings.filterwarnings('ignore', message='CUDA initialization')

# Keep numpy / torch from spawning internal threads (parallelism comes only from parallel_map processes)
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '1')

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

torch.manual_seed(0)
np.random.seed(0)

dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device('cpu')          # matrices are small, so CPU is faster
torch.set_default_device(device)
torch.set_num_threads(1)


# Number of parallel processes (override with the IRT_WORKERS environment variable)
def _detect_workers():
    override = os.environ.get('IRT_WORKERS')
    if override:
        return max(1, int(override))
    try:
        n = len(os.sched_getaffinity(0))
    except AttributeError:
        n = os.cpu_count() or 4
    return max(1, min(n, 32))


SEARCH_WORKERS = _detect_workers()
print(f'SEARCH_WORKERS = {SEARCH_WORKERS}')

# **2. Load data**

In [ ]:
_pisa_df = pd.read_csv(DATA_PATH)            # rows = items, columns = students
X_pisa_raw = torch.tensor(_pisa_df.values, dtype=dtype, device=device)
X_pisa_raw = torch.transpose(X_pisa_raw, 0, 1)   # -> students x items
mask_nan_pisa = ~torch.isnan(X_pisa_raw)
X_pisa = torch.where(mask_nan_pisa, X_pisa_raw, torch.zeros_like(X_pisa_raw))
PISA_N, PISA_J = X_pisa.shape

# Known clusters: items (1-17)(18-35)(36-52)(53-68)(69-83)(84-102) -> 0-indexed
PISA_KNOWN_CLUSTERS = [range(0, 17), range(17, 35), range(35, 52), range(52, 68), range(68, 83), range(83, 102)]
assert sum(len(c) for c in PISA_KNOWN_CLUSTERS) == PISA_J, (
    f"number of cluster items ({sum(len(c) for c in PISA_KNOWN_CLUSTERS)}) != J ({PISA_J})")

edge_mask_pisa = torch.zeros(PISA_J, PISA_J)
for cl in PISA_KNOWN_CLUSTERS:
    idx = torch.tensor(list(cl))
    edge_mask_pisa[idx.unsqueeze(1), idx.unsqueeze(0)] = 1.0
edge_mask_pisa.fill_diagonal_(0.0)

print(X_pisa.shape, "observed mean:", X_pisa[mask_nan_pisa].mean().item(),
      "  missing rate:", (~mask_nan_pisa).float().mean().item())

# **3. Function definitions**

Common utilities: parallel execution, cell-level splits, learning-rate schedule, handling S

In [ ]:
def _worker_init():
    import warnings
    warnings.filterwarnings('ignore', message='CUDA initialization')
    import torch
    torch.set_default_dtype(torch.float64)
    torch.set_num_threads(1)


# Run several fits across processes. Each fit has its own seed, so results do not depend on parallelism.
def parallel_map(fn, items, max_workers=None):
    items = list(items)
    if len(items) > 1:
        try:
            from joblib.externals.loky import get_reusable_executor
            pool = get_reusable_executor(max_workers=max_workers or SEARCH_WORKERS,
                                         initializer=_worker_init, timeout=600)
            return list(pool.map(fn, items))
        except (ImportError, RuntimeError, OSError, BlockingIOError) as e:
            print(f'    ! parallel execution failed ({type(e).__name__}: {e}) -> falling back to sequential.')
    return [fn(x) for x in items]


# Randomly hold out a fraction frac of the observed cells (per cell, not per person)
def split_cells(mask, frac, gen=None):
    rand = torch.rand(mask.shape, dtype=torch.float32, device=mask.device, generator=gen)
    mask_held = mask & (rand < frac)
    mask_keep = mask & (rand >= frac)
    return mask_keep, mask_held


TEST_FRAC = 0.15          # train 85 / test 15


# Split observed cells into train / test (no validation set)
def make_splits(mask_nan, gen=None):
    mask_train, mask_test = split_cells(mask_nan, frac=TEST_FRAC, gen=gen)
    return dict(train_fit=mask_train, test_final=mask_test)


# Adam learning rate: linear warm-up for LR_WARMUP steps, halved after LR_PLATEAU_PATIENCE consecutive checks without improvement
LR_WARMUP, LR_PLATEAU_PATIENCE, LR_DECAY, LR_FLOOR_FRAC = 25, 4, 0.5, 0.05


class LRSchedule:
    def __init__(self, optimizer):
        self.scale, self.stall = 1.0, 0
        self._sched = torch.optim.lr_scheduler.LambdaLR(optimizer, self._factor)

    def _factor(self, t):
        return min((t + 1) / LR_WARMUP, 1.0) * self.scale

    def step(self):
        self._sched.step()

    def on_val(self, improved):
        if improved:
            self.stall = 0
            return
        self.stall += 1
        if self.stall % LR_PLATEAU_PATIENCE == 0:
            self.scale = max(self.scale * LR_DECAY, LR_FLOOR_FRAC)


S_INIT_SCALE = 0.01       # scale of the initial s_edge
A_INIT_SCALE = 1.0        # scale of the initial A


def init_s_edge(n, gen=None):
    return S_INIT_SCALE * torch.randn(n, generator=gen)


# S is learned only through the upper-triangular edge vector s_edge (zero diagonal, symmetric)
def build_symmetric_S(s_edge, J, iu, ju):
    S = torch.zeros(J, J, dtype=s_edge.dtype, device=s_edge.device)
    if s_edge.numel() > 0:
        S[iu, ju] = s_edge
        S[ju, iu] = s_edge
    return S


# Indices of candidate edges (i<j). If edge_mask is given, only allowed pairs
def edge_index(J, edge_mask=None):
    iu, ju = torch.triu_indices(J, J, offset=1)
    if edge_mask is not None:
        keep = edge_mask[iu, ju] > 0
        iu, ju = iu[keep], ju[keep]
    return iu, ju


# Resolve sign indeterminacy: flip A and theta together so that column means are positive
def orient_sign(A, Theta):
    signs = torch.sign(A.mean(dim=0))
    signs[signs == 0] = 1.0
    D = torch.diag(signs)
    return A @ D, Theta @ D

VIF-model (theta estimated by an encoder)

In [ ]:
N_MC_SAMPLES = 5          # theta samples drawn per iteration


# encoder: responses -> posterior mean and log-variance of theta.
# Input is [X, mask] (2J dims): responses (0 if unobserved) and observation indicators. Item parameters A, d, S are not inputs.
class ThetaEncoder(nn.Module):
    def __init__(self, J, K, hidden=32):
        super().__init__()
        self.K = K
        self.net = nn.Sequential(
            nn.Linear(2 * J, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2 * K),
        )

    def build_input(self, X_filled, observed_mask):
        return torch.cat([X_filled, observed_mask.to(X_filled.dtype)], dim=1)

    def forward(self, enc_input):
        out = self.net(enc_input)
        mu, log_var = out[:, :self.K], out[:, self.K:]
        log_var = torch.clamp(log_var, min=-8.0, max=8.0)
        return mu, log_var


# Redraw encoder weights from this fit's own RNG (same distribution as the default nn.Linear init)
def reseed_encoder_(encoder, gen):
    with torch.no_grad():
        for m in encoder.modules():
            if isinstance(m, nn.Linear):
                bound = float(m.weight.shape[1]) ** -0.5
                m.weight.uniform_(-bound, bound, generator=gen)
                if m.bias is not None:
                    m.bias.uniform_(-bound, bound, generator=gen)
    return encoder


def score_theta_vif(X, observed_mask, A, encoder):
    with torch.no_grad():
        X_filled = torch.where(observed_mask, X, torch.zeros_like(X))
        mu, _ = encoder(encoder.build_input(X_filled, observed_mask))
    return mu


# Integrals over theta are computed by quadrature (K=1 only).
#   GH_Z, GH_W       : Gauss-Hermite nodes for expectations under q = N(mu, sigma^2) (theta = mu + sigma * z)
#   TH_GRID, TH_LOGW : equally spaced grid (step 0.05) and log weights for integrating against the prior N(0, 1)
_gh_z, _gh_w = np.polynomial.hermite_e.hermegauss(21)
GH_Z, GH_W = torch.tensor(_gh_z), torch.tensor(_gh_w / _gh_w.sum())
TH_GRID = torch.linspace(-8.0, 8.0, 321)
TH_LOGW = -0.5 * TH_GRID ** 2 - 0.5 * np.log(2 * np.pi) + np.log(float(TH_GRID[1] - TH_GRID[0]))


# Sum of NLL over observed cells for each person x theta node.  theta: (n, Q),  offset = X S - d: (n, J)  ->  (n, Q)
def nll_on_theta_points(theta, A, offset, X_zero, obs_f):
    assert A.shape[1] == 1, 'theta quadrature supports K=1 only'
    logits = theta.unsqueeze(-1) * A[:, 0] + offset.unsqueeze(1)
    nll = F.binary_cross_entropy_with_logits(logits, X_zero.unsqueeze(1).expand_as(logits), reduction='none')
    return (nll * obs_f.unsqueeze(1)).sum(dim=-1)


# E_q[NLL] summed over persons (q = N(mu, std^2))
def expected_nll(mu, std, A, offset, X_zero, obs_f):
    return float((nll_on_theta_points(mu + std * GH_Z, A, offset, X_zero, obs_f) * GH_W).sum())


# Marginal NLL = - sum_i log ∫ exp(-NLL_i(theta)) N(theta; 0, 1) dtheta.  Used as the fit term of BIC.
def marginal_nll(A, offset, X_zero, obs_f, chunk=500):
    total = 0.0
    for a in range(0, X_zero.shape[0], chunk):
        sl = slice(a, a + chunk)
        nll = nll_on_theta_points(TH_GRID.expand(X_zero[sl].shape[0], -1), A, offset[sl], X_zero[sl], obs_f[sl])
        total -= float(torch.logsumexp(TH_LOGW - nll, dim=1).sum())
    return total


# VIF-model:  logits = theta A' - d + X S,  theta ~ q(theta | x) = N(mu, sigma^2) (encoder)
#   objective Q = (E_q[NLL] + KL) / N + gamma * |s_edge|_1
#   A, d, encoder use Adam; s_edge takes an lr_S step along the gradient from the same backward pass, then soft-threshold (threshold lr_S * gamma)
#   early stopping: every check_every steps, evaluate Q on the train cells (E_q from fixed MC draws); after patience checks without improvement, restore the best state
#   if init is given, start from that solution (warm start). If init has no matching s_edge, S starts at 0.
#   fit_graph=False gives S = 0 (graph-free MIRT)
def fit_vif(X, observed_mask, K, gamma=0.0, fit_graph=True, edge_mask=None,
            n_iter=1500, lr=0.05, lr_S=0.05, hidden=16, check_every=25, patience=16,
            init=None, gen=None):
    N, J = X.shape
    iu, ju = (edge_index(J, edge_mask) if fit_graph
              else (torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)))

    if init is None:
        # A starts positive. Starting with mixed signs can trap the fit in a local solution with split loading signs (not a constraint).
        A = (A_INIT_SCALE * torch.randn(J, K, generator=gen)).abs().requires_grad_(True)
        d = torch.zeros(J, requires_grad=True)
        encoder = reseed_encoder_(ThetaEncoder(J, K, hidden=hidden).to(dtype=X.dtype), gen)
        s_edge = init_s_edge(iu.numel(), gen).requires_grad_(True)
    else:
        A = init['A'].clone().requires_grad_(True)
        d = init['d'].clone().requires_grad_(True)
        encoder = ThetaEncoder(J, K, hidden=hidden).to(dtype=X.dtype)
        encoder.load_state_dict(init['encoder'])
        s_prev = init.get('s_edge')
        s_edge = ((s_prev.clone() if (s_prev is not None and s_prev.numel() == iu.numel())
                   else torch.zeros(iu.numel())).requires_grad_(True))

    X_zero = torch.where(observed_mask, X, torch.zeros_like(X))
    obs_f = observed_mask.to(X.dtype)

    # 0/1 mask of which loadings are estimated. All loadings are free for now (change this for a confirmatory structure).
    Lambda_mask = torch.ones(J, K)

    def _A_eff(A_):
        return A_ * Lambda_mask

    optimizer = torch.optim.Adam([A, d] + list(encoder.parameters()), lr=lr)
    scheduler = LRSchedule(optimizer)
    best_loss, best_state, stall = float('inf'), None, 0
    # Noise for the early-stopping theta draws. Drawn once and reused at every check (so comparisons between checks are free of MC noise)
    check_eps = torch.randn(N_MC_SAMPLES, N, K, dtype=X.dtype, generator=gen)

    for t in range(n_iter):
        optimizer.zero_grad(set_to_none=True)
        S_sym = build_symmetric_S(s_edge, J, iu, ju)

        mu, log_var = encoder(encoder.build_input(X_zero, observed_mask))
        std = (0.5 * log_var).exp()
        mc_eps = torch.randn(N_MC_SAMPLES, *mu.shape, dtype=mu.dtype, generator=gen)
        theta_samples = mu.unsqueeze(0) + std.unsqueeze(0) * mc_eps

        logits_mc = theta_samples @ _A_eff(A).T - d + (X_zero @ S_sym).unsqueeze(0)
        nll_elem = F.binary_cross_entropy_with_logits(
            logits_mc, X_zero.unsqueeze(0).expand_as(logits_mc), reduction='none')
        nll_sum = (nll_elem * obs_f.unsqueeze(0)).sum() / N_MC_SAMPLES
        kl = 0.5 * (std ** 2 + mu ** 2 - 1 - log_var).sum()

        loss = (nll_sum + kl) / N
        loss.backward()
        optimizer.step()
        scheduler.step()

        # s_edge: plain gradient step + soft-threshold (the L1 penalty is not differentiated; it is applied only here)
        if fit_graph and s_edge.numel() > 0:
            with torch.no_grad():
                gs = s_edge.grad
                if gs is not None:
                    z = s_edge - lr_S * gs
                    s_edge.copy_(torch.sign(z) * torch.clamp(z.abs() - lr_S * gamma, min=0.0))
                    s_edge.grad = None

        # Early stopping: stop when Q = E_q[NLL] + KL (+ penalty) on the train cells stops improving. E_q is the average over draws built from check_eps
        if t % check_every == 0:
            with torch.no_grad():
                S_cur = build_symmetric_S(s_edge, J, iu, ju)
                mu_v, lv_v = encoder(encoder.build_input(X_zero, observed_mask))
                theta_v = mu_v.unsqueeze(0) + (0.5 * lv_v).exp().unsqueeze(0) * check_eps
                logits_v = theta_v @ _A_eff(A).T - d + (X_zero @ S_cur).unsqueeze(0)
                cur_loss = (F.binary_cross_entropy_with_logits(
                    logits_v, X_zero.unsqueeze(0).expand_as(logits_v), reduction='none')
                    * obs_f.unsqueeze(0)).sum().item() / N_MC_SAMPLES
                cur_loss += (0.5 * (lv_v.exp() + mu_v ** 2 - 1 - lv_v).sum().item()
                             + N * gamma * s_edge.abs().sum().item())
            scheduler.on_val(cur_loss < best_loss)
            if cur_loss < best_loss:
                best_loss = cur_loss
                best_state = (A.detach().clone(), s_edge.detach().clone(), d.detach().clone(),
                              {k: v.detach().clone() for k, v in encoder.state_dict().items()})
                stall = 0
            else:
                stall += 1
                if stall >= patience:
                    break

    A, s_edge, d, enc_state = best_state
    encoder.load_state_dict(enc_state)

    with torch.no_grad():
        A = _A_eff(A)
        S_final = build_symmetric_S(s_edge, J, iu, ju)
        mu_final, log_var_final = encoder(encoder.build_input(X_zero, observed_mask))
        std_final = (0.5 * log_var_final).exp()
        kl_final = 0.5 * (std_final ** 2 + mu_final ** 2 - 1 - log_var_final).sum()
        offset_final = X_zero @ S_final - d
        nll_final = expected_nll(mu_final, std_final, A, offset_final, X_zero, obs_f)
        Q_final = ((nll_final + kl_final) / N + gamma * s_edge.abs().sum()).item()
        marg_nll = marginal_nll(A, offset_final, X_zero, obs_f)

    diag = dict(n_edges=int((s_edge.abs() > 1e-3).sum().item()),
                n_iter_run=t + 1,
                marg_nll=marg_nll,      # marginal NLL with theta integrated out (sum over persons). Used by BIC
                # solution that can be passed as init to another fit (warm start)
                state=dict(A=A.detach().clone(), d=d.detach().clone(),
                           s_edge=s_edge.detach().clone(),
                           encoder={k: v.detach().clone() for k, v in encoder.state_dict().items()}))

    return A.detach(), S_final.detach(), mu_final.detach(), d.detach(), Q_final, encoder, diag


# Covariance of the theta actually used by the encoder model = Cov(mu) + E[sigma^2]. Square roots of the diagonal should be near 1 for A to be on the right scale.
def encoder_theta_cov(X, mask, A_raw, encoder, theta_oriented=None):
    with torch.no_grad():
        X_filled = torch.where(mask, X, torch.zeros_like(X))
        mu, log_var = encoder(encoder.build_input(X_filled, mask))
        rows = mask.any(dim=1)
        m = (mu if theta_oriented is None else theta_oriented)[rows]
        mc = m - m.mean(dim=0, keepdim=True)
        cov_mu = (mc.T @ mc) / max(mc.shape[0] - 1, 1)
        return cov_mu + torch.diag(log_var[rows].exp().mean(dim=0))


def encoder_total_theta_sd(X, mask, A_raw, encoder):
    return float(torch.diagonal(encoder_theta_cov(X, mask, A_raw, encoder)).sqrt().mean().item())

FLaG-IRT (model with theta integrated out)

In [ ]:
# FLaG-IRT (Chen et al., 2018): model with theta integrated out.  M = A Sigma A' + S,  logit_j = sum_{i != j} M_ij x_i + M_jj / 2
def build_sigma(B_free, K):
    if K == 1:
        return torch.ones(1, 1)
    B = torch.eye(K)
    tril_idx = torch.tril_indices(K, K, offset=-1)
    B = B.clone()
    B[tril_idx[0], tril_idx[1]] = B_free
    return B @ B.T


def pseudo_neg_loglik_flag(X, observed_mask, A, S, Sigma):
    X_filled = torch.where(observed_mask, X, torch.zeros_like(X))
    L = A @ Sigma @ A.T
    S_sym = 0.5 * (S + S.T)
    M = L + S_sym
    diagM = torch.diagonal(M)
    logits = X_filled @ M + diagM.unsqueeze(0) * (0.5 - X_filled)
    nll_elem = F.binary_cross_entropy_with_logits(logits, X_filled, reduction='none')
    return (nll_elem * observed_mask).sum()


# Build logits from context cells and score only on target cells
def pseudo_neg_loglik_flag_ctx(X, ctx_mask, tgt_mask, A, S, Sigma):
    X_ctx = torch.where(ctx_mask, X, torch.zeros_like(X))
    M = A @ Sigma @ A.T + 0.5 * (S + S.T)
    diagM = torch.diagonal(M)
    logits = X_ctx @ M + diagM.unsqueeze(0) * (0.5 - X_ctx)
    nll_elem = F.binary_cross_entropy_with_logits(logits, X, reduction='none')
    return (nll_elem * tgt_mask).sum()


# Same update scheme as VIF: A, Sigma, diag(S) use Adam; s_edge takes an lr_S step + soft-threshold.
# Early stopping: convergence of the training objective (including the penalty).
# init passes on only A / diag(S) / Sigma; s_edge is passed on only if its length matches (otherwise it starts at 0).
def fit_flag(X, observed_mask, K, gamma=0.0, fit_graph=True, edge_mask=None,
             n_iter=1500, lr=0.05, lr_S=0.05, check_every=25, patience=16,
             init=None, gen=None):
    N, J = X.shape
    iu, ju = (edge_index(J, edge_mask) if fit_graph
              else (torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)))
    A = (A_INIT_SCALE * torch.randn(J, K, generator=gen)).abs().requires_grad_(True)     # start positive, as in VIF
    s_diag = torch.zeros(J).requires_grad_(True)
    s_edge = init_s_edge(iu.numel(), gen).requires_grad_(True)
    if K > 1:
        B_free = torch.zeros(K * (K - 1) // 2, requires_grad=True)
    else:
        B_free = None
    if init is not None:
        with torch.no_grad():
            A.copy_(init['A'])
            s_diag.copy_(init['S_diag'])
            if B_free is not None and init.get('B_free') is not None:
                B_free.copy_(init['B_free'])
            if init.get('s_edge') is not None and init['s_edge'].numel() == s_edge.numel():
                s_edge.copy_(init['s_edge'])
            else:
                s_edge.zero_()

    smooth_params = ([A, s_diag] if B_free is None else [A, B_free, s_diag])
    optimizer = torch.optim.Adam(smooth_params, lr=lr)
    scheduler = LRSchedule(optimizer)
    best_loss, best_state, stall = float('inf'), None, 0

    def _build_S(s_e, s_d):
        return build_symmetric_S(s_e, J, iu, ju) + torch.diag(s_d)

    for t in range(n_iter):
        optimizer.zero_grad(set_to_none=True)
        Sigma = build_sigma(B_free, K)
        S = _build_S(s_edge, s_diag)
        loss = pseudo_neg_loglik_flag(X, observed_mask, A, S, Sigma) / N
        loss.backward()
        optimizer.step()
        scheduler.step()

        if fit_graph and s_edge.numel() > 0:
            with torch.no_grad():
                gs = s_edge.grad
                if gs is not None:
                    z = s_edge - lr_S * gs
                    s_edge.copy_(torch.sign(z) * torch.clamp(z.abs() - lr_S * gamma, min=0.0))
                    s_edge.grad = None

        if t % check_every == 0:
            with torch.no_grad():
                Sigma_cur = build_sigma(B_free, K)
                cur_loss = pseudo_neg_loglik_flag_ctx(
                    X, observed_mask, observed_mask, A, _build_S(s_edge, s_diag), Sigma_cur).item() / N
                cur_loss += gamma * s_edge.abs().sum().item()
            scheduler.on_val(cur_loss < best_loss)
            if cur_loss < best_loss:
                best_loss = cur_loss
                best_state = (A.detach().clone(), s_diag.detach().clone(),
                              s_edge.detach().clone(),
                              B_free.detach().clone() if B_free is not None else None)
                stall = 0
            else:
                stall += 1
                if stall >= patience:
                    break

    A, s_diag, s_edge, B_free_best = best_state
    if B_free is not None:
        B_free = B_free_best

    with torch.no_grad():
        S_final = _build_S(s_edge, s_diag)
        Sigma_final = build_sigma(B_free, K)
        loss_final = (pseudo_neg_loglik_flag(X, observed_mask, A, S_final, Sigma_final).item() / N
                      + gamma * float(s_edge.abs().sum()))
    state_out = dict(A=A.detach().clone(), S_diag=s_diag.detach().clone(),
                     B_free=B_free.detach().clone() if B_free is not None else None,
                     n_iter_run=t + 1)
    return A.detach(), S_final.detach(), Sigma_final.detach(), float(loss_final), state_out


# Map the FLaG solution to the VIF representation (A, d, S_off, theta). Exactly equal to the original formula on the scored cells.
#   A_c = A chol(Sigma),  theta = X_ctx A_c (linear score),  d = -diag(M)/2,  S_off = off-diagonal of S
def flag_to_common(A, S, Sigma, X, ctx_mask):
    S_sym = 0.5 * (S + S.T)
    M = A @ Sigma @ A.T + S_sym
    A_c = A @ torch.linalg.cholesky(Sigma)
    S_off = S_sym - torch.diag(torch.diagonal(S_sym))
    d = -0.5 * torch.diagonal(M).clone()
    theta = torch.where(ctx_mask, X, torch.zeros_like(X)) @ A_c
    return A_c, d, S_off, theta

Evaluation metrics

In [ ]:
# Binary AUC (Mann-Whitney U, ties get average ranks)
def roc_auc_binary(y_true, y_score):
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    order = np.argsort(y_score, kind='mergesort')
    sorted_scores = y_score[order]
    n = len(sorted_scores)
    ranks = np.empty(n)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + 1 + j + 1) / 2.0
        i = j + 1
    sum_ranks_pos = ranks[y_true == 1].sum()
    return float((sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


# Binary PR-AUC (average precision)
# The algorithm follows scikit-learn's average_precision_score / _binary_clf_curve
# (https://github.com/scikit-learn/scikit-learn, BSD-3-Clause License)
def average_precision_binary(y_true, y_score):
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    n = len(y_true)
    order = np.argsort(y_score, kind='mergesort')[::-1]
    y_true_sorted = y_true[order]
    y_score_sorted = y_score[order]
    distinct = np.where(np.diff(y_score_sorted))[0]
    threshold_idxs = np.r_[distinct, n - 1]
    tps = np.cumsum(y_true_sorted)[threshold_idxs].astype(np.float64)
    fps = (1 + threshold_idxs - tps).astype(np.float64)
    ps = tps + fps
    precision = np.divide(tps, ps, out=np.zeros_like(tps), where=(ps != 0))
    total_pos = tps[-1]
    recall = np.ones_like(tps) if total_pos == 0 else tps / total_pos
    precision = np.r_[precision[::-1], 1.0]
    recall = np.r_[recall[::-1], 0.0]
    return float(-np.sum(np.diff(recall) * precision[:-1]))


# Compare the support of S with a reference graph S_true: precision / recall / F1 / PR-AUC etc.
def s_support_metrics(S_hat, S_true, tol=1e-3):
    J = S_hat.shape[0]
    iu, ju = torch.triu_indices(J, J, offset=1)
    S_sym = 0.5 * (S_hat + S_hat.T)
    score = S_sym[iu, ju].abs().detach().cpu().numpy()
    truth = (S_true[iu, ju].abs() > 1e-12).cpu().numpy().astype(int)
    pred = (score > tol).astype(int)
    tp = int((pred & truth).sum())
    fp = int((pred & (1 - truth)).sum())
    fn = int(((1 - pred) & truth).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    pr_auc = float(average_precision_binary(truth, score)) if truth.sum() and truth.sum() < len(truth) else float('nan')
    deg = (S_sym.abs() > tol).sum(dim=1)
    return {'S_precision': prec, 'S_recall': rec, 'S_f1': f1, 'S_pr_auc': pr_auc,
            'S_nedges': int(pred.sum()), 'S_maxdeg': int(deg.max().item()),
            'S_rmse': torch.sqrt(((S_sym - S_true) ** 2)[iu, ju].mean()).item()}


# Held-out performance: build logits from context cells and measure AUC / NLL on test cells
def heldout_metrics(X, theta_test, A_or, d, S_sym, mask_context, mask_test):
    with torch.no_grad():
        X_context = torch.where(mask_context, X, torch.zeros_like(X))
        logits = theta_test @ A_or.T - d + X_context @ S_sym
        nll = (F.binary_cross_entropy_with_logits(logits, X, reduction='none')
               * mask_test).sum().item()
        n_cells = int(mask_test.sum().item())
        probs = torch.sigmoid(logits)[mask_test].cpu().numpy()
        labels = X[mask_test].cpu().numpy()
    auc = float(roc_auc_binary(labels, probs)) if len(np.unique(labels)) > 1 else float('nan')
    return {'test_auc': auc, 'test_nll': nll, 'test_nll_per_cell': nll / max(n_cells, 1)}


# Chandrasekaran-Parrilo-Willsky (2012) identifiability index: how identifiable the low-rank + sparse decomposition is (no ground truth needed)
#   cpw_inc   = sqrt(max leverage of col(A))    how concentrated the low-rank part is on a few items
#   cpw_mu    = largest eigenvalue of the support adjacency matrix    how clustered the sparse part is
#   cpw_ident = inc x mu                          larger means A and S can more easily substitute for each other (lower bound of the paper's xi*mu condition)
def cpw_ident_metrics(A, S, tol=1e-3):
    J = A.shape[0]
    iu, ju = torch.triu_indices(J, J, offset=1)
    with torch.no_grad():
        S_sym = 0.5 * (S + S.T)
        sup = S_sym[iu, ju].abs() > tol
        U = torch.linalg.svd(A, full_matrices=False)[0]
        inc = float((U ** 2).sum(dim=1).max().sqrt())
        Adj = torch.zeros(J, J, dtype=A.dtype, device=A.device)
        Adj[iu[sup], ju[sup]] = 1.0
        Adj = Adj + Adj.T
        mu = float(torch.linalg.eigvalsh(Adj).max()) if bool(sup.any()) else 0.0
    return {'cpw_inc': inc, 'cpw_mu': mu, 'cpw_ident': inc * mu}


# "mean ± SD" table by group
def mean_sd_table(df, by, cols, digits=3):
    g = df.groupby(by, sort=False)[cols]
    m, s = g.mean().round(digits), g.std().round(digits)
    return m.astype(str) + ' ± ' + s.astype(str)

Model fitting: every model returns results in the same format

In [ ]:
LR = 0.05                 # Adam learning rate for all fits
# Graph models and their "graph-free" counterparts. The counterpart's solution is the warm start and the reference for detecting A collapse.
REF_OF = {'VIF-model': 'MIRT-only', 'FLaG': 'FLaG-nograph'}


# Fit one model and return it in the common format: A, S (symmetric, zero diagonal), d, theta_test, theta_train, diag, state
#   name : 'VIF-model' / 'MIRT-only' (VIF without S) / 'FLaG' / 'FLaG-nograph' (FLaG without off-diagonal S)
#   Every model trains on train_fit cells only and stops on convergence of the training objective. theta is estimated with train_fit cells as context.
def run_model(name, X, splits, K, edge_mask=None, gamma=0.0, n_iter=800, lr=LR, hidden=16,
              init=None, fit_seed=None):
    gen = torch.Generator(device=device).manual_seed(int(fit_seed))
    mtf = splits['train_fit']
    t0 = time.time()

    if name in ('FLaG', 'FLaG-nograph'):
        A_raw, S_raw, Sigma, Q_final, fstate = fit_flag(
            X, mtf, K, gamma=gamma, fit_graph=(name == 'FLaG'), edge_mask=edge_mask,
            n_iter=n_iter, lr=lr, init=init, gen=gen)
        A_c, d, S_sym, theta_train = flag_to_common(A_raw, S_raw, Sigma, X, mtf)
        signs = torch.sign(A_c.mean(dim=0))
        signs[signs == 0] = 1.0
        D_sgn = torch.diag(signs)
        A_or, theta_train = A_c @ D_sgn, theta_train @ D_sgn
        theta_test = torch.where(mtf, X, torch.zeros_like(X)) @ A_or
        diag = dict(n_edges=int((torch.triu(S_sym.abs(), 1) > 1e-3).sum().item()),
                    # FLaG theta is a linear score, so its sd is not 1 (not subject to the scale check)
                    theta_total_sd=float(theta_train[mtf.any(dim=1)].std(dim=0).mean().item()),
                    Q_final=float(Q_final), state=fstate,
                    flag_A=A_raw.detach(), flag_Sigma=Sigma.detach(),
                    flag_S_diag=torch.diagonal(S_raw).detach().clone(),
                    n_iter_run=fstate['n_iter_run'])

    else:
        A_raw, S, mu, d, Q_final, enc, diag = fit_vif(
            X, mtf, K, gamma=gamma, fit_graph=(name != 'MIRT-only'), edge_mask=edge_mask,
            n_iter=n_iter, lr=lr, hidden=hidden, init=init, gen=gen)
        S_sym = 0.5 * (S + S.T)
        A_or, _ = orient_sign(A_raw, mu)
        _, theta_train = orient_sign(A_raw, score_theta_vif(X, mtf, A_raw, enc))
        theta_test = theta_train
        diag['theta_total_sd'] = encoder_total_theta_sd(X, mtf, A_raw, enc)
        diag['Q_final'] = float(Q_final)

    return dict(name=name, A=A_or, S=S_sym, d=d,
                theta_test=theta_test, theta_train=theta_train, diag=diag,
                state=diag.get('state'))

Unknown graph: FLaG selects by BIC, VIF-model refits on that support (+ gamma path diagnostics)

In [ ]:
GAMMA_FRAC = (0.10, 1.05)   # gamma grid range (relative to gamma_max)
N_GAMMA = 20                # number of grid points
N_REF_STARTS = 5            # number of graph-free VIF-model reference fits with different seeds (the one with the lowest training Q is used)
SUPPORT_TOL = 0.01          # in regularized fits, only pairs with |s| above this count as support (excludes values flickering near the threshold)
if SMOKE:
    N_GAMMA = 14
    N_REF_STARTS = 2


def edge_support(S_sym, tol=1e-3):
    J = S_sym.shape[0]
    iu, ju = torch.triu_indices(J, J, offset=1)
    return (S_sym[iu, ju].abs() > tol).cpu().numpy()


# Smallest gamma at which the L1 term turns the graph off entirely at S = 0: |d(NLL/N)/dS|_inf. Used to scale the gamma grid.
def gamma_max_of(name, X, splits, ref, K):
    N, J = X.shape
    mtf = splits['train_fit']
    X_zero = torch.where(mtf, X, torch.zeros_like(X))
    iu, ju = torch.triu_indices(J, J, offset=1)
    s0 = torch.zeros(iu.numel(), requires_grad=True)
    S_off = build_symmetric_S(s0, J, iu, ju)

    if name == 'FLaG':
        S_tot = S_off + torch.diag(ref['flag_S_diag'])
        nll = pseudo_neg_loglik_flag(X, mtf, ref['flag_A'], S_tot, ref['flag_Sigma']) / N
    else:
        base = ref['theta_train'] @ ref['A'].T - ref['d']
        logits = base + X_zero @ S_off
        nll = (F.binary_cross_entropy_with_logits(logits, X_zero, reduction='none')
               * mtf.to(X.dtype)).sum() / N
    return float(torch.autograd.grad(nll, s0)[0].abs().max().item())


# Graph-free solution: warm start for every candidate, scale of the gamma grid, reference for detecting A collapse
#   Depending on initialization, the VIF-model reference fit (MIRT-only) can end in a solution where theta is switched off (A ~ 0). So we fit
#   N_REF_STARTS seeds and keep the lowest training Q (the first start uses fit_seed as is). FLaG is fitted once.
def graph_free_reference(name, X, splits, K, n_iter, fit_seed=0, hidden=16):
    seeds = [fit_seed + 7919 * k for k in range(N_REF_STARTS if name == 'VIF-model' else 1)]

    def _start(seed):
        return run_model(REF_OF[name], X, splits, K, gamma=0.0, n_iter=n_iter, lr=LR, hidden=hidden,
                         fit_seed=seed)

    fits = parallel_map(_start, seeds, max_workers=SEARCH_WORKERS)
    res = min(fits, key=lambda r: r['diag']['Q_final'])
    res['diag']['ref_starts_Q'] = [r['diag']['Q_final'] for r in fits]      # training Q per start (for inspection)
    ref = dict(res=res, state=res['state'], Q=res['diag']['Q_final'],
               A_norm=float(torch.linalg.norm(res['A'])),
               A=res['A'], d=res['d'], theta_train=res['theta_train'])
    ref.update({k: res['diag'].get(k) for k in ('flag_A', 'flag_Sigma', 'flag_S_diag')})
    ref['gamma_max'] = gamma_max_of(name, X, splits, ref, K)
    return ref


# Dedicated seed per candidate
def _fit_seed_for(gamma, tag=0):
    return (int(round(float(gamma) * 1e7)) * 131 + int(tag) * 7919) % (2 ** 31)


def gamma_grid_of(ref):
    return np.geomspace(GAMMA_FRAC[0] * ref['gamma_max'],
                        GAMMA_FRAC[1] * ref['gamma_max'], N_GAMMA)


# Number of parameters: loadings J*K + intercepts J + edges (+ K(K-1)/2 for Sigma in FLaG)
def n_params_of(name, J, K, n_edges):
    n = J * K + J + int(n_edges)
    return n + (K * (K - 1) // 2 if name == 'FLaG' else 0)


def support_to_mask(support, J):
    iu, ju = torch.triu_indices(J, J, offset=1)
    v = torch.as_tensor(support, dtype=dtype)
    m = torch.zeros(J, J, dtype=dtype)
    m[iu, ju] = v
    m[ju, iu] = v
    return m


# Gamma path for one model (procedure of Chen et al., 2018). Returns (path, sel)
#   1) L1-regularized fit at each gamma on the grid (warm start from the graph-free solution ref) -> support {|s_ij| > SUPPORT_TOL}
#   2) for each distinct support, an unpenalized refit allowing only that support
#   3) for each candidate: BIC = 2 NLL_refit + (number of parameters) log N,  |A|/|A_ref|,  cpw_ident
#      NLL_refit (sum over persons): for VIF the marginal NLL with theta integrated by quadrature, for FLaG the objective itself (N Q_refit)
#   4) sel = minimum BIC over all candidates (ties go to the sparser one).
#      No gate filters candidates: the |A| ratio and cpw_ident are only reported, and admissible is always True.
def gamma_path(name, X, splits, K, n_iter, ref, fit_seed=0, hidden=16):
    N, J = X.shape
    n_persons = int(splits['train_fit'].any(dim=1).sum().item())
    grid = gamma_grid_of(ref)
    kw = dict(n_iter=n_iter, lr=LR, hidden=hidden)

    def _reg(g):
        r = run_model(name, X, splits, K, edge_mask=None, gamma=float(g),
                      init=ref['state'], fit_seed=_fit_seed_for(g, fit_seed), **kw)
        return dict(gamma=float(g), reg=r, support=edge_support(r['S'], tol=SUPPORT_TOL))

    path = sorted(parallel_map(_reg, grid, max_workers=SEARCH_WORKERS), key=lambda p: p['gamma'])

    # Refit each distinct support only once
    uniq = {}
    for p in path:
        uniq.setdefault(p['support'].tobytes(), p)

    def _refit(p):
        mask = support_to_mask(p['support'], J)
        iu, ju = edge_index(J, mask)
        init = dict(p['reg']['state'])
        init['s_edge'] = p['reg']['S'][iu, ju].clone()
        r = run_model(name, X, splits, K, edge_mask=mask, gamma=0.0, init=init,
                      fit_seed=_fit_seed_for(p['gamma'], fit_seed + 1), **kw)
        return p['support'].tobytes(), r

    refits = dict(parallel_map(_refit, list(uniq.values()), max_workers=SEARCH_WORKERS))

    for p in path:
        r = refits[p['support'].tobytes()]
        p['n_edges'] = int(p['support'].sum())
        p['Q_refit'] = r['diag']['Q_final']
        p['nll_refit'] = r['diag'].get('marg_nll', N * p['Q_refit'])
        p['bic'] = 2.0 * p['nll_refit'] + n_params_of(name, J, K, p['n_edges']) * np.log(n_persons)
        p['A_norm_ratio'] = float(torch.linalg.norm(r['A'])) / max(ref['A_norm'], 1e-12)
        p['cpw_ident'] = cpw_ident_metrics(r['A'], r['S'])['cpw_ident']
        p['admissible'] = True
        del p['reg']

    sel = min(path, key=lambda p: (p['bic'], -p['gamma']))      # ties go to the sparser one
    for p in path:
        p['selected'] = p is sel
    return path, sel


PATH_COLS = ('gamma', 'n_edges', 'Q_refit', 'nll_refit', 'bic', 'A_norm_ratio', 'cpw_ident',
             'admissible', 'selected')


def path_table(path):
    return [{k: p[k] for k in PATH_COLS} for p in path]


def print_path(name, ref, path, sel, full=True, note='BIC selection'):
    N = len(ref['theta_train'])
    if full:
        print(f"    [{name}] gamma_max={ref['gamma_max']:.5f}  grid [{path[0]['gamma']:.5f}, {path[-1]['gamma']:.5f}] x {len(path)}")
        for p in path:
            print(f"      gamma={p['gamma']:.5f}  edges={p['n_edges']:4d}  Q_refit={p['Q_refit']:.4f}  NLL_refit/N={p['nll_refit'] / N:.4f}  "
                  f"BIC={p['bic']:10.1f}  |A|/|A_ref|={p['A_norm_ratio']:4.2f}  cpw_ident={p['cpw_ident']:5.2f}"
                  f"{'  <- BIC minimum' if p is sel else ''}")
    print(f"    [{name}] {note} gamma={sel['gamma']:.5f}  edges={sel['n_edges']}  BIC={sel['bic']:.1f}  "
          f"({len({p['support'].tobytes() for p in path})} distinct supports / {len(path)} grid points)")


# Unknown graph, BIC selection (used by FLaG). Returns (gamma, ref, path, res)
#   graph-free solution -> gamma path -> minimum-BIC support -> refit from the graph-free solution with that support as the given graph
def select_gamma_bic(name, X, splits, K, n_iter, fit_seed=0, hidden=16, verbose=True):
    J = X.shape[1]
    ref = graph_free_reference(name, X, splits, K, n_iter, fit_seed=fit_seed, hidden=hidden)
    path, sel = gamma_path(name, X, splits, K, n_iter, ref, fit_seed=fit_seed, hidden=hidden)
    res = run_model(name, X, splits, K, edge_mask=support_to_mask(sel['support'], J),
                    gamma=0.0, init=ref['state'], fit_seed=fit_seed, n_iter=n_iter, lr=LR, hidden=hidden)
    res['diag'].update(gamma_sel=sel['gamma'], bic=sel['bic'], n_edges_sel=sel['n_edges'], support=sel['support'])
    print_path(name, ref, path, sel, full=verbose)
    return sel['gamma'], ref, path_table(path), res


# Unknown graph for VIF-model: refit with the support selected by FLaG's BIC as the given graph. Returns (nan, ref, path, res)
#   No re-selection on the VIF side (the |A| ratio and cpw_ident appear in the results table).
#   Warm start from the graph-free VIF solution (same as the known-graph arm).
#   with_path=True also computes and returns VIF's own gamma path for diagnostics (not used for selection). Otherwise path is an empty list.
def vif_on_flag_support(X, splits, K, n_iter, flag_res, with_path=False, fit_seed=0, hidden=16, verbose=True):
    J = X.shape[1]
    support = flag_res['diag']['support']
    ref = graph_free_reference('VIF-model', X, splits, K, n_iter, fit_seed=fit_seed, hidden=hidden)
    res = run_model('VIF-model', X, splits, K, edge_mask=support_to_mask(support, J),
                    gamma=0.0, init=ref['state'], fit_seed=fit_seed, n_iter=n_iter, lr=LR, hidden=hidden)
    res['diag'].update(gamma_sel=float('nan'), n_edges_sel=int(support.sum()))
    table = []
    if with_path:
        path, sel = gamma_path('VIF-model', X, splits, K, n_iter, ref, fit_seed=fit_seed, hidden=hidden)
        print_path('VIF-model', ref, path, sel, full=verbose, note='(diagnostic path; not used for selection) BIC minimum')
        res['diag'].update(own_bic_support=sel['support'])      # support VIF's own BIC would pick (used by the comparison arm)
        table = path_table(path)
    return float('nan'), ref, table, res


# Call the unknown arm of both models the same way. FLaG must be called first and its result passed as flag_res.
def select_unknown(name, X, splits, K, n_iter, flag_res=None, with_path=False, fit_seed=0, hidden=16, verbose=True):
    if name == 'FLaG':
        return select_gamma_bic(name, X, splits, K, n_iter, fit_seed=fit_seed, hidden=hidden, verbose=verbose)
    return vif_on_flag_support(X, splits, K, n_iter, flag_res, with_path=with_path,
                               fit_seed=fit_seed, hidden=hidden, verbose=verbose)


# Gamma path diagnostic plot: one panel per model, y = BIC, x = gamma, grey dotted line = number of edges at that gamma.
#   tables: {model name: gamma path table (output of path_table or its DataFrame)}
def plot_gamma_paths(tables, title):
    tables = {k: pd.DataFrame(v).sort_values('gamma') for k, v in tables.items() if len(v)}
    if not tables:
        return
    fig, axes = plt.subplots(1, len(tables), figsize=(6.4 * len(tables), 3.6), squeeze=False)
    for j, (name, t) in enumerate(tables.items()):
        g, ok = t['gamma'].to_numpy(), t['admissible'].to_numpy(dtype=bool)
        for i, (col, lab) in enumerate([('bic', 'BIC')]):
            ax, y = axes[i, j], t[col].to_numpy()
            ax.plot(g, y, '-', color='C0', lw=1)
            ax.plot(g[ok], y[ok], 'o', color='C0', ms=4, label='candidate')
            if (~ok).any():
                ax.plot(g[~ok], y[~ok], 'x', color='C3', ms=6, label='excluded by gate')
            for gs in t.loc[t['selected'].astype(bool), 'gamma']:
                ax.axvline(float(gs), color='k', ls='--', lw=0.8, label='BIC minimum')
            ax.set_xscale('log')
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))     # ticks like 0.01
            ax.xaxis.set_minor_formatter(plt.NullFormatter())
            ax.set_ylabel(lab)
            ax.grid(alpha=0.3)
            ax2 = ax.twinx()
            ax2.plot(g, t['n_edges'].to_numpy(), ':', color='0.55', lw=1.2)
            ax2.set_ylabel('number of edges (grey dotted)', color='0.45', fontsize=8)
            ax.set_title(name, fontsize=10)
            ax.legend(fontsize=7)
            ax.set_xlabel('gamma')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

# **4. PISA experiment: fit known / unknown / MIRT-only over repeated splits**

In [ ]:
PISA_K = 1               # PISA_N_SEEDS and PISA_N_ITER are set in the configuration cell at the top

PISA_GRAPH_MODELS = ['VIF-model', 'FLaG']
# Splits on which VIF-model's own gamma path (and the comparison arm) is computed. Currently all; list a subset such as (0, 4) to save time
PISA_PATH_SPLITS = tuple(range(PISA_N_SEEDS))

# Only the **split seed** changes between repetitions: which cells are used for training and which are held out for test.
t0 = time.time()
pisa_runs, pisa_splits, pisa_paths, pisa_refs, pisa_gamma = [], [], {}, {}, {}     # pisa_runs[s] = {arm name: fit result}
pisa_path_rows = []        # gamma paths of all splits (for the comparison figures)
for s in range(PISA_N_SEEDS):
    splits_s = make_splits(mask_nan_pisa, gen=torch.Generator(device=device).manual_seed(20260919 + s))
    run_s = {}
    # unknown: FLaG selects by BIC, VIF-model refits on the support FLaG selected (hence FLaG goes first).
    # On splits in PISA_PATH_SPLITS, VIF-model's gamma path is also computed (not used by the unknown arm; used by the comparison arm and figures).
    for name in ('FLaG', 'VIF-model'):
        i = PISA_GRAPH_MODELS.index(name)
        gam_s, ref_s, path_s, res_unk = select_unknown(
            name, X_pisa, splits_s, PISA_K, PISA_N_ITER, flag_res=run_s.get('FLaG (unknown graph)'),
            with_path=(s in PISA_PATH_SPLITS), fit_seed=8100, hidden=16, verbose=(s == 0))
        # known: clusters are given as the graph. Warm start from the reference fit of the same split.
        res_kn = run_model(
            name, X_pisa, splits_s, PISA_K, edge_mask=edge_mask_pisa,
            gamma=0.0, n_iter=PISA_N_ITER, lr=0.05, hidden=16, fit_seed=8200 + i,
            init=ref_s['state'])
        for r in (res_kn, res_unk):
            r['diag']['A_norm_ratio'] = float(torch.linalg.norm(r['A'])) / ref_s['A_norm']
        run_s[f'{name} (known graph)'], run_s[f'{name} (unknown graph)'] = res_kn, res_unk
        if name == 'VIF-model':
            ref_s['res']['diag']['A_norm_ratio'] = 1.0
            run_s['MIRT-only'] = ref_s['res']               # same fit as the reference (reused)
        if s == 0:
            pisa_gamma[name], pisa_refs[name], pisa_paths[name] = gam_s, ref_s, path_s
        pisa_path_rows += [dict(split=s, model=name, **p) for p in path_s]
        if name == 'VIF-model' and s in PISA_PATH_SPLITS:
            # Comparison: arm fitted on the support VIF-model selects by its own BIC (minimum BIC of the diagnostic path)
            res_own = run_model(name, X_pisa, splits_s, PISA_K,
                                edge_mask=support_to_mask(res_unk['diag']['own_bic_support'], PISA_J),
                                gamma=0.0, n_iter=PISA_N_ITER, lr=0.05, hidden=16, fit_seed=8100, init=ref_s['state'])
            res_own['diag']['A_norm_ratio'] = float(torch.linalg.norm(res_own['A'])) / ref_s['A_norm']
            run_s['VIF-model (own BIC)'] = res_own
        print(f'  split {s} [{name}] done  ({time.time() - t0:.0f}s)')
    pisa_runs.append(run_s)
    pisa_splits.append(splits_s)
    print(f'  split {s}: ' + ', '.join(f'{k} edges={v["diag"]["n_edges"]}' for k, v in run_s.items()))
pisa_results = pisa_runs[0]                                 # split 0 wherever a single result is needed
pisa_seed_res = {name: [r[f'{name} (unknown graph)'] for r in pisa_runs] for name in PISA_GRAPH_MODELS}
df_pisa_path = pd.DataFrame(pisa_path_rows)

# **5. Results**

Performance table: mean ± SD, per-split values, gamma path

In [ ]:
# PISA performance table. No ground truth, so report held-out performance, cluster agreement and identifiability.
def _pisa_row(res, splits):
    row = heldout_metrics(X_pisa, res['theta_test'], res['A'], res['d'], res['S'],
                          splits['train_fit'], splits['test_final'])
    row.update(cpw_ident_metrics(res['A'], res['S']))
    row.update(S_nedges=res['diag']['n_edges'], A_norm_ratio=res['diag']['A_norm_ratio'],
               theta_total_sd=res['diag']['theta_total_sd'], n_iter_run=res['diag'].get('n_iter_run'),
               gamma=res['diag'].get('gamma_sel', float('nan')))
    if res['diag']['n_edges'] > 0:
        m = s_support_metrics(res['S'], edge_mask_pisa)
        row.update({k: m[k] for k in ('S_precision', 'S_recall', 'S_f1', 'S_maxdeg')})
    return row


# all arms x all splits
df_pisa_all = pd.DataFrame([dict(model=name, split=s, **_pisa_row(res, pisa_splits[s]))
                            for s, run_s in enumerate(pisa_runs) for name, res in run_s.items()])
df_pisa = df_pisa_all[df_pisa_all.split == 0].set_index('model')     # split 0
_pc = ['test_auc', 'test_nll_per_cell', 'S_nedges', 'S_precision', 'S_recall', 'cpw_inc', 'cpw_mu', 'cpw_ident',
       'A_norm_ratio', 'theta_total_sd', 'n_iter_run']
print(f'=== PISA (known + unknown): mean ± SD ({PISA_N_SEEDS} split seeds; data and fitting seeds fixed) ===')
print(mean_sd_table(df_pisa_all, 'model', _pc, digits=4).to_string())
print(f'(within-cluster pairs in the known graph = {int(edge_mask_pisa.sum().item()) // 2}; '
      f'n_iter_run = {PISA_N_ITER} means the cap was hit before convergence)')
print('S metrics measure agreement with the item clusters, not accuracy of true edge recovery. Read with the co-observation diagnostic below.')
print('cpw_ident = inc x mu (CPW identifiability index). Larger means A and S substitute for each other more easily; not used for selection.')
print('For FLaG, theta_total_sd is not expected to be 1. SDs reflect variation across train/test splits '
      '(train and test cells change together).')

print('\n=== per-split values ===')
print(df_pisa_all.set_index(['model', 'split'])[
    ['test_auc', 'test_nll_per_cell', 'S_nedges', 'S_precision', 'cpw_ident', 'A_norm_ratio', 'gamma', 'n_iter_run']]
      .round(4).to_string())

print('\n=== gamma paths of the unknown arms (split 0) ===')
for name in PISA_GRAPH_MODELS:
    print(f'[{name}]')
    print(pd.DataFrame(pisa_paths[name]).round(4).to_string(index=False))

Co-observation diagnostic: how much do the S metrics overlap with the booklet design?

*Diagnostic only.* Separating real local dependence from the booklet design needs further experiments; the numbers below describe the overlap and are not used to claim that local dependence was (or was not) found.

* `rho_co` = spearman(|S|, number of students who saw both items). Lower means less of the design is absorbed.
* `co_at_k` = precision when the same number of edges is chosen by co-observation count alone. `vs_design = precision / co_at_k` near 1 means the model merely restates the design.
* `lift_i` = within co-observation stratum i, (same-cluster share of selected edges) / (base rate of that stratum). Above 1 may point to signal not explained by the design.

In [ ]:
_M = mask_nan_pisa.to(dtype)
_co_mat = _M.T @ _M                                   # number of students who saw both items i and j
_iu, _ju = torch.triu_indices(PISA_J, PISA_J, 1)
_truth = (edge_mask_pisa[_iu, _ju] > 0).cpu().numpy().astype(int)
_co = _co_mat[_iu, _ju].cpu().numpy()
_base = _truth.mean()
_STRATA = np.unique(np.percentile(_co, [0, 25, 50, 75, 90, 100]))


def _spearman(a, b):
    return float(np.corrcoef(pd.Series(a).rank(), pd.Series(b).rank())[0, 1])


def _prec_at_k(score, k):
    return float(_truth[np.argsort(-score, kind='mergesort')[:k]].mean()) if k else float('nan')


print(f'mean co-observation: same-cluster pairs {_co[_truth==1].mean():.0f} students / '
      f'different-cluster pairs {_co[_truth==0].mean():.0f} students')
print(f'PR-AUC for predicting clusters from co-observation counts alone = '
      f'{average_precision_binary(_truth, _co):.3f}   (random baseline {_base:.3f})')
print()

_rows = []
for _name, _res in pisa_results.items():
    if _name == 'MIRT-only':
        continue
    _s = (0.5 * (_res['S'] + _res['S'].T))[_iu, _ju].abs().cpu().numpy()
    _sel = _s > 1e-3
    _k = int(_sel.sum())
    _r = dict(model=_name, k=_k, rho_co=_spearman(_s, _co) if _k else float('nan'),
              precision=float(_truth[_sel].mean()) if _k else float('nan'),
              co_at_k=_prec_at_k(_co, _k), chance=_base)
    _r['vs_design'] = _r['precision'] / _r['co_at_k'] if _r['co_at_k'] else float('nan')
    for _i, (_lo, _hi) in enumerate(zip(_STRATA[:-1], _STRATA[1:])):
        _m = (_co >= _lo) & ((_co < _hi) if _hi != _STRATA[-1] else (_co <= _hi))
        _b = _truth[_m].mean()
        _ne = int((_sel & _m).sum())
        _r[f'lift{_i}'] = (_truth[_sel & _m].mean() / _b) if (_ne >= 10 and _b > 0) else np.nan
    _rows.append(_r)

df_pisa_confound = pd.DataFrame(_rows).set_index('model')
print('=== co-observation confounding diagnostic (diagnostic only; not a test for local dependence) ===')
print(df_pisa_confound.round(3).to_string())
print()
print(f'  stratum boundaries (co-observation): {np.round(_STRATA, 0).tolist()}')
print('  rho_co    = spearman(|S|, co-observation). Lower = less of the design absorbed.')
print('  vs_design = precision / co_at_k. Near 1 = merely restates the design.')
print('  lift_i    = within stratum i, (same-cluster share of selected edges) / (base rate of that stratum).')

S heatmap: unknown-graph mean over splits (left = no boxes, right = PISA item clusters shown as coloured boxes)

In [ ]:
cluster_colors_pisa = plt.get_cmap('tab10').colors
_iup, _jup = torch.triu_indices(PISA_J, PISA_J, offset=1)


def _pisa_freq(name):
    """Number of splits in which each edge was selected (symmetric matrix)."""
    F_ = torch.zeros(PISA_J, PISA_J)
    for res in pisa_seed_res[name]:
        F_ += (res['S'].abs() > 1e-3).to(F_.dtype)
    return F_.fill_diagonal_(0.0)


def _pisa_mean_S(name):
    """Mean estimated S over splits (splits where the edge was not selected contribute 0)."""
    return torch.stack([res['S'] for res in pisa_seed_res[name]]).mean(dim=0)


def _draw_pisa(ax, S, title, boxes=True):
    S_vis = S.clone()
    S_vis.fill_diagonal_(0.0)
    vmax = S_vis.abs().max().item() or 1.0
    im = ax.imshow(S_vis.cpu().numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Item')
    ax.set_ylabel('Item')
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for k, cl in enumerate(PISA_KNOWN_CLUSTERS if boxes else []):
        a, w = cl.start, len(cl)
        ax.add_patch(plt.Rectangle((a - 0.5, a - 0.5), w, w, fill=False,
                                   edgecolor=cluster_colors_pisa[k % 10], linewidth=1.8))


# Draw each model's split-averaged unknown-graph S twice: left without boxes, right with item-cluster boxes overlaid
fig, axes = plt.subplots(len(PISA_GRAPH_MODELS), 2, figsize=(12, 5.4 * len(PISA_GRAPH_MODELS)))
for row, name in zip(np.atleast_2d(axes), PISA_GRAPH_MODELS):
    _mS = _pisa_mean_S(name)
    _draw_pisa(row[0], _mS, f'{name} unknown graph', boxes=False)
    _draw_pisa(row[1], _mS, f'{name} unknown graph (item clusters)')
for _ax in np.atleast_2d(axes)[:-1].flat:      # only the bottom row keeps its x label (it collides with the next title)
    _ax.set_xlabel('')
legend_handles = [mpatches.Patch(facecolor='none', edgecolor=cluster_colors_pisa[k % 10], linewidth=1.8,
                                 label=f"cluster {k + 1}: items {cl.start + 1}-{cl.stop}")
                  for k, cl in enumerate(PISA_KNOWN_CLUSTERS)]
fig.legend(handles=legend_handles, loc='lower center', ncol=3, frameon=False, fontsize=9)
fig.suptitle(f'PISA — mean estimated S over {PISA_N_SEEDS} splits, unknown graph (box colours = PISA item clusters)')
fig.tight_layout(rect=(0, 0.07, 1, 0.97))
plt.show()

_same = (edge_mask_pisa[_iup, _jup] > 0).cpu().numpy()
for name in PISA_GRAPH_MODELS:
    _f = _pisa_freq(name)[_iup, _jup].cpu().numpy()
    print(f'[{name}] ' + '  '.join(
        f'>= {k}/{PISA_N_SEEDS}: {int((_f >= k).sum())} edges (same cluster {int(((_f >= k) & _same).sum())})'
        for k in range(1, PISA_N_SEEDS + 1)))

Gamma path diagnostic: BIC on the first split for each model (FLaG = path used for selection, VIF-model = diagnostic only)

In [ ]:
plot_gamma_paths({name: pisa_paths[name] for name in PISA_GRAPH_MODELS}, 'PISA — gamma path (split 0)')

Comparison figures (FLaG vs the exploratory VAE-based variants)

In [ ]:
# Shared tools for the comparison figures. x = arm, faint dots = individual splits, horizontal tick and bar = mean ± SD over splits.
ARG_STYLE = {      # arm -> (label used in figures, colour)
    'VIF-model (known graph)': ('VIF-model\n(known)', 'C0'),
    'VIF-model (unknown graph)': ('VIF-model\n(FLaG-selected\nsupport)', 'C1'),
    'VIF-model (own BIC)': ('VIF-model\n(own-BIC\nsupport)', 'C4'),
    'FLaG (known graph)': ('FLaG\n(known)', 'C2'),
    'FLaG (unknown graph)': ('FLaG\n(unknown, BIC)', 'C3'),
    'MIRT-only': ('MIRT-only\n(VIF without\ngraph)', '0.45'),
}


# truth: {metric name: true value} draws a black horizontal line in that panel
def arg_strip(df, arms, metrics, title, truth=None):
    fig, axes = plt.subplots(1, len(metrics), figsize=(max(4.2, 1.35 * len(arms)) * len(metrics), 4.4), squeeze=False)
    for ax, (col, lab) in zip(axes[0], metrics):
        for i, arm in enumerate(arms):
            name, color = ARG_STYLE[arm]
            v = df[df['model'] == arm][col].to_numpy(dtype=float)
            v = v[~np.isnan(v)]
            if len(v) == 0:
                continue
            ax.plot(np.full(len(v), i), v, 'o', color=color, ms=4, alpha=0.35, ls='none')
            ax.errorbar(i, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else 0.0, fmt='_', color=color, ms=18, capsize=3, lw=1.5)
        if truth and col in truth:
            ax.axhline(truth[col], color='k', lw=1.0)
        ax.set_xticks(range(len(arms)))
        ax.set_xticklabels([ARG_STYLE[a][0] for a in arms], fontsize=7)
        ax.set_xlim(-0.6, len(arms) - 0.4)
        ax.set_title(lab, fontsize=10)
        ax.grid(alpha=0.3, axis='y')
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()

ARG_METRICS = [('test_auc', 'held-out AUC (higher is better)'),
               ('test_nll_per_cell', 'held-out NLL per cell (lower is better)'),
               ('cpw_ident', 'CPW identifiability index')]

# Comparison 1: known graph (item clusters) — FLaG vs VIF-model vs graph-free MIRT (PISA has no ground truth, so only held-out performance and identifiability)
arg_strip(df_pisa_all, ['VIF-model (known graph)', 'FLaG (known graph)', 'MIRT-only'], ARG_METRICS,
          'PISA — known graph (item clusters): FLaG vs VIF-model vs VIF without graph (points = splits, bars = mean ± SD)')

In [ ]:
# Comparison 2-1: how much does the support selected by each model's own BIC vary across splits (FLaG vs VIF-model)?
#   S_precision is the share of selected edges inside an item cluster (agreement with the test design, not a true graph).
arg_strip(df_pisa_all, ['VIF-model (own BIC)', 'FLaG (unknown graph)'],
          [('S_nedges', 'number of selected edges'), ('S_precision', 'share of selected edges inside item clusters')],
          "PISA — support selected by each model's own BIC, across splits (points = splits, bars = mean ± SD)")

print('Number of edges selected by own BIC: mean / SD over splits')
print(df_pisa_all[df_pisa_all['model'].isin(['VIF-model (own BIC)', 'FLaG (unknown graph)'])]
      .groupby('model')['S_nedges'].agg(['mean', 'std']).round(1).to_string())

# One BIC curve per repetition. x = number of edges, y = BIC relative to the sparsest candidate.
#   large dot = candidate selected by BIC in that repetition
def arg_bic_panel(ax, d, key, title, n_true=None):
    for r, t in d.groupby(key):
        sel_edges = t.loc[t['selected'].astype(bool), 'n_edges'].tolist()
        t = t.drop_duplicates('n_edges').sort_values('n_edges')
        e, y, ok = t['n_edges'].to_numpy(), t['bic'].to_numpy() - float(t['bic'].iloc[0]), t['admissible'].to_numpy(dtype=bool)
        c = f'C{int(r) % 10}'
        ax.plot(e, y, '-', color=c, lw=1, alpha=0.8)
        ax.plot(e[~ok], y[~ok], 'x', color=c, ms=5)
        ax.plot(e[np.isin(e, sel_edges)], y[np.isin(e, sel_edges)], 'o', color=c, ms=7)
    if n_true is not None:
        ax.axvline(n_true, color='k', ls=':', lw=1.0)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('number of edges')
    ax.set_ylabel('BIC - BIC of the sparsest candidate')
    ax.grid(alpha=0.3)


# Comparison 2-2: shape of the BIC curves (one line = one split)
fig, axes = plt.subplots(1, len(PISA_GRAPH_MODELS), figsize=(6.2 * len(PISA_GRAPH_MODELS), 3.9), squeeze=False)
for j, name in enumerate(PISA_GRAPH_MODELS):
    arg_bic_panel(axes[0, j], df_pisa_path[df_pisa_path.model == name], 'split', name)
fig.suptitle('PISA — BIC curves per split (one line = one split, large dot = BIC minimum)', y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# Comparison 3: how does VIF-model refitted on the FLaG-selected support compare with FLaG and the other arms?
_arms3 = ['VIF-model (unknown graph)', 'VIF-model (own BIC)', 'VIF-model (known graph)', 'FLaG (unknown graph)', 'MIRT-only']
arg_strip(df_pisa_all, _arms3, ARG_METRICS,
          'PISA — VIF-model refitted on the FLaG-selected support vs alternatives (points = splits, bars = mean ± SD)')

print('Arm means:')
print(df_pisa_all[df_pisa_all['model'].isin(_arms3)].groupby('model')[
    [c for c, _ in ARG_METRICS] + ['S_nedges', 'S_precision']].mean().round(4).to_string())